In [11]:
# ============================================================
# FULL PRESENTATION-READY CODE
# Advanced 3D Multiphase Wound Healing Simulation
# with Migration + Proliferation Visualization
# Google Colab ready
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import gridspec
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# ------------------------------------------------------------
# 1. Parameters
# ------------------------------------------------------------
N = 90                  # grid size
dx = 1.0
dt = 0.12
steps = 720
save_every = 6

# Epithelial closure
D_u = 0.22
r_u = 0.065
gamma_B = 0.05         # vascular support helps closure
gamma_I = 0.03         # inflammation slightly slows closure

# Inflammation
D_I = 0.30
lambda_I = 0.018

# Angiogenic signal (VEGF-like)
D_V = 0.24
beta_V = 0.18
lambda_V = 0.030

# Vascular density
D_B = 0.05
r_B = 0.22
lambda_B = 0.008

# Scar remodeling
k_S = 0.030
lambda_S = 0.002

# Initial wound geometry
wound_radius = 0.34    # normalized radius

# ------------------------------------------------------------
# 2. Spatial grid
# ------------------------------------------------------------
x = np.linspace(-1, 1, N)
y = np.linspace(-1, 1, N)
X, Y = np.meshgrid(x, y)
R = np.sqrt(X**2 + Y**2)
Theta = np.arctan2(Y, X)

W0 = (R < wound_radius).astype(float)   # initial wound mask

# ------------------------------------------------------------
# 3. Initial fields
# ------------------------------------------------------------
# u: epithelial cell density
u = np.ones((N, N))
u[W0 == 1] = 0.0

# I: inflammation
I = np.zeros((N, N))
I[W0 == 1] = 1.0

# V: angiogenic signal
V = np.zeros((N, N))

# B: vascular density
B = np.zeros((N, N))

# S: scar/collagen density
S = np.zeros((N, N))

# small noise outside wound
np.random.seed(42)
u += 0.01 * np.random.rand(N, N)
u = np.clip(u, 0, 1)
u[W0 == 1] = 0.0

# ------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------
def laplacian(Z, dx=1.0):
    """
    2D Laplacian with no-flux boundary condition.
    """
    Zp = np.pad(Z, 1, mode='edge')
    return (
        Zp[2:, 1:-1] + Zp[:-2, 1:-1] +
        Zp[1:-1, 2:] + Zp[1:-1, :-2] -
        4 * Zp[1:-1, 1:-1]
    ) / dx**2

def smooth_field(A, passes=5):
    """
    Simple iterative smoothing.
    """
    B = A.copy()
    for _ in range(passes):
        B = (
            B +
            np.roll(B, 1, axis=0) +
            np.roll(B, -1, axis=0) +
            np.roll(B, 1, axis=1) +
            np.roll(B, -1, axis=1)
        ) / 5.0
    return B

# ------------------------------------------------------------
# 5. Static texture fields
# ------------------------------------------------------------
np.random.seed(7)

# Skin texture
skin_noise1 = smooth_field(np.random.rand(N, N), passes=14)
skin_noise2 = smooth_field(np.random.rand(N, N), passes=5)

skin_base = np.array([0.87, 0.72, 0.62])

skin_rgb = np.zeros((N, N, 3))
for c in range(3):
    skin_rgb[:, :, c] = skin_base[c]

skin_rgb += 0.08 * (skin_noise1[..., None] - 0.5)
skin_rgb += 0.03 * (skin_noise2[..., None] - 0.5)
skin_rgb = np.clip(skin_rgb, 0, 1)

# Wound bed texture
wound_noise = smooth_field(np.random.rand(N, N), passes=9)
wound_base = np.array([0.77, 0.25, 0.28])

wound_rgb = np.zeros((N, N, 3))
for c in range(3):
    wound_rgb[:, :, c] = wound_base[c]

wound_rgb += 0.14 * (wound_noise[..., None] - 0.5)
wound_rgb = np.clip(wound_rgb, 0, 1)

# Scar texture
scar_noise = smooth_field(np.random.rand(N, N), passes=8)
scar_base = np.array([0.92, 0.83, 0.79])

scar_rgb = np.zeros((N, N, 3))
for c in range(3):
    scar_rgb[:, :, c] = scar_base[c]

scar_rgb += 0.03 * (scar_noise[..., None] - 0.5)
scar_rgb = np.clip(scar_rgb, 0, 1)

# Vessel-like branching template
branch_raw = (
    np.cos(8 * Theta + 16 * R + 2.5 * skin_noise1) +
    0.6 * np.cos(13 * Theta - 10 * R + 1.8 * skin_noise2)
)
branch_raw = np.clip(branch_raw, 0, None)

ring_pref = np.exp(-((R - (wound_radius + 0.10)) ** 2) / 0.06)
vascular_template = smooth_field(branch_raw * ring_pref, passes=2)
if vascular_template.max() > 0:
    vascular_template /= vascular_template.max()

skin_relief = smooth_field(np.random.rand(N, N), passes=10) - 0.5

# ------------------------------------------------------------
# 6. Simulation
# ------------------------------------------------------------
frames = []
wound_area = []

for step in range(steps):

    # epithelial closure
    du = (
        D_u * laplacian(u, dx)
        + r_u * u * (1 - u)
        + gamma_B * B * (1 - u)
        - gamma_I * I * u
    )

    # inflammation
    dI = (
        D_I * laplacian(I, dx)
        + 0.22 * (1 - u) * W0
        - lambda_I * I
    )

    # angiogenic signal
    dV = (
        D_V * laplacian(V, dx)
        + beta_V * I * (1 - u)
        - lambda_V * V
    )

    # vascular density
    dB = (
        D_B * laplacian(B, dx)
        + r_B * V * (0.25 + 0.75 * vascular_template) * (1 - B)
        - lambda_B * B
    )

    # scar remodeling
    healed_zone = np.clip((u - 0.75) / 0.25, 0, 1)
    dS = (
        k_S * W0 * healed_zone * (1 - np.clip(I, 0, 1))
        - lambda_S * S
    )

    # update fields
    u += dt * du
    I += dt * dI
    V += dt * dV
    B += dt * dB
    S += dt * dS

    # clip
    u = np.clip(u, 0, 1)
    I = np.clip(I, 0, 1.5)
    V = np.clip(V, 0, 1.5)
    B = np.clip(B, 0, 1)
    S = np.clip(S, 0, 1)

    wound_area.append(np.sum(u < 0.5))

    if step % save_every == 0:
        frames.append((u.copy(), I.copy(), B.copy(), S.copy()))

print(f"Simulation finished. Saved {len(frames)} frames.")

# ------------------------------------------------------------
# 7. Rendering and mechanism fields
# ------------------------------------------------------------
def compute_phase(u, I, S):
    wound_fraction = np.mean(1 - u[W0 == 1])
    infl = np.mean(I[W0 == 1])
    scar = np.mean(S[W0 == 1])

    if wound_fraction > 0.75 and infl > 0.35:
        return "Inflammatory phase"
    elif wound_fraction > 0.40:
        return "Proliferative phase"
    elif wound_fraction > 0.12:
        return "Closure phase"
    elif scar > 0.15:
        return "Scar remodeling phase"
    else:
        return "Late healing phase"

def render_frame(u, I, B, S):
    us = smooth_field(u, passes=2)
    Is = smooth_field(I, passes=2)
    Bs = smooth_field(B, passes=1)
    Ss = smooth_field(S, passes=2)

    # 3D height map
    Z = 0.020 * skin_relief
    Z += -0.46 * (1 - us)              # wound depression
    Z += 0.05 * Ss * W0                # scar slight elevation
    Z += -0.04 * Is * (1 - us)         # inflammatory distortion
    Z = smooth_field(Z, passes=1)

    # base color blend
    alpha = us[..., None]
    rgb = alpha * skin_rgb + (1 - alpha) * wound_rgb

    # inflammation halo
    infl = np.clip(Is / 1.2, 0, 1)[..., None]
    infl_color = np.array([0.95, 0.36, 0.28])
    rgb = (1 - 0.20 * infl) * rgb + 0.20 * infl * infl_color

    # healing rim
    gx = np.roll(us, -1, axis=1) - np.roll(us, 1, axis=1)
    gy = np.roll(us, -1, axis=0) - np.roll(us, 1, axis=0)
    grad = np.sqrt(gx**2 + gy**2)
    if grad.max() > 0:
        grad = grad / grad.max()

    rim = np.clip(grad * 3.0, 0, 1)[..., None]
    rim_color = np.array([0.75, 0.12, 0.14])
    rgb = (1 - 0.18 * rim) * rgb + 0.18 * rim * rim_color

    # angiogenesis / vessel-like overlay
    vessel_map = np.clip(Bs * vascular_template * 1.8, 0, 1)[..., None]
    vessel_color = np.array([0.63, 0.07, 0.09])
    rgb = (1 - 0.28 * vessel_map) * rgb + 0.28 * vessel_map * vessel_color

    # scar overlay
    scar_map = np.clip(Ss * W0, 0, 1)[..., None]
    rgb = (1 - 0.45 * scar_map) * rgb + 0.45 * scar_map * scar_rgb

    rgb = np.clip(rgb, 0, 1)
    return Z, rgb

def compute_migration_field(u):
    """
    Epithelial migration visualized as diffusive flux:
    J = -D_u * grad(u)
    """
    us = smooth_field(u, passes=2)
    du_dx = (np.roll(us, -1, axis=1) - np.roll(us, 1, axis=1)) / (2 * dx)
    du_dy = (np.roll(us, -1, axis=0) - np.roll(us, 1, axis=0)) / (2 * dx)
    Jx = -D_u * du_dx
    Jy = -D_u * du_dy
    return Jx, Jy

def compute_proliferation_field(u):
    """
    Proliferation intensity from logistic growth term.
    """
    us = smooth_field(u, passes=2)
    P = r_u * us * (1 - us)
    return P

# ------------------------------------------------------------
# 8. Set up professional figure layout
# ------------------------------------------------------------
fig = plt.figure(figsize=(14, 7.6))
gs = gridspec.GridSpec(1, 3, width_ratios=[1.25, 1.0, 0.06], wspace=0.18)

ax3d = fig.add_subplot(gs[0, 0], projection='3d')
ax2d = fig.add_subplot(gs[0, 1])
cax = fig.add_subplot(gs[0, 2])

# Fixed color scale for proliferation
prolif_norm = Normalize(vmin=0, vmax=r_u * 0.25)
sm = ScalarMappable(norm=prolif_norm, cmap='YlOrRd')
sm.set_array([])

cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Proliferation intensity", fontsize=10)
cbar.ax.tick_params(labelsize=9)

# ------------------------------------------------------------
# 9. Animation update function
# ------------------------------------------------------------
def update(frame_idx):
    ax3d.clear()
    ax2d.clear()

    u_f, I_f, B_f, S_f = frames[frame_idx]
    Z, colors = render_frame(u_f, I_f, B_f, S_f)
    phase = compute_phase(u_f, I_f, S_f)

    # wound closure percentage
    current_wound_area = np.sum(u_f < 0.5)
    initial_wound_area = np.sum(W0 == 1)
    wound_remaining = 100 * current_wound_area / initial_wound_area
    wound_closed = 100 - wound_remaining

    # ---------------- LEFT PANEL: 3D SURFACE ----------------
    ax3d.plot_surface(
        X, Y, Z,
        facecolors=colors,
        rstride=1,
        cstride=1,
        linewidth=0,
        antialiased=True,
        shade=False
    )

    ax3d.set_title(
        "3D Multiphase Skin Wound Healing Simulation\n"
        f"Frame {frame_idx} | {phase} | Wound closure: {wound_closed:.1f}%",
        fontsize=12,
        pad=18
    )

    ax3d.set_xlim(-1, 1)
    ax3d.set_ylim(-1, 1)
    ax3d.set_zlim(-0.60, 0.15)

    ax3d.set_xlabel("x-position across tissue surface\n(normalized units)", fontsize=9, labelpad=10)
    ax3d.set_ylabel("y-position across tissue surface\n(normalized units)", fontsize=9, labelpad=10)
    ax3d.set_zlabel("relative tissue surface height", fontsize=9, labelpad=10)

    ax3d.set_xticks([-1, -0.5, 0, 0.5, 1])
    ax3d.set_yticks([-1, -0.5, 0, 0.5, 1])
    ax3d.set_zticks([-0.5, -0.25, 0])

    ax3d.tick_params(axis='both', which='major', labelsize=8)
    ax3d.tick_params(axis='z', which='major', labelsize=8)

    ax3d.set_box_aspect((1, 1, 0.35))
    ax3d.view_init(elev=42, azim=45)
    ax3d.grid(True)

    ax3d.text2D(
        0.02, 0.94,
        "Model variables:\n"
        "u = epithelial cell density\n"
        "I = inflammation signal\n"
        "B = vascular support\n"
        "S = scar remodeling",
        transform=ax3d.transAxes,
        fontsize=8.5,
        verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.82)
    )

    ax3d.text2D(
        0.67, 0.06,
        "Color interpretation:\n"
        "red = open/inflamed wound\n"
        "dark red = angiogenesis\n"
        "pale region = scar tissue\n"
        "skin tone = healed tissue",
        transform=ax3d.transAxes,
        fontsize=8.5,
        verticalalignment="bottom",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.82)
    )

    # ---------------- RIGHT PANEL: MECHANISMS ----------------
    P = compute_proliferation_field(u_f)
    Jx, Jy = compute_migration_field(u_f)

    im = ax2d.imshow(
        P,
        extent=[-1, 1, -1, 1],
        origin='lower',
        cmap='YlOrRd',
        norm=prolif_norm
    )

    # subsample arrows
    step_q = 6
    Xq = X[::step_q, ::step_q]
    Yq = Y[::step_q, ::step_q]
    Jxq = Jx[::step_q, ::step_q]
    Jyq = Jy[::step_q, ::step_q]

    mag = np.sqrt(Jxq**2 + Jyq**2)
    mask = mag > 0.002

    ax2d.quiver(
        Xq[mask], Yq[mask],
        Jxq[mask], Jyq[mask],
        color='cyan',
        angles='xy',
        scale_units='xy',
        scale=0.08,
        width=0.006,
        alpha=0.9
    )

    # wound boundary contour
    wound_edge = (u_f < 0.5).astype(float)
    ax2d.contour(
        X, Y, wound_edge,
        levels=[0.5],
        colors='black',
        linewidths=2
    )

    ax2d.set_title(
        "Mechanistic Visualization:\nEpithelial Migration and Proliferation",
        fontsize=12
    )

    ax2d.set_xlabel("x-position across tissue surface (normalized units)", fontsize=10)
    ax2d.set_ylabel("y-position across tissue surface (normalized units)", fontsize=10)

    ax2d.set_xlim(-1, 1)
    ax2d.set_ylim(-1, 1)
    ax2d.set_aspect('equal')
    ax2d.grid(True, alpha=0.25)

    ax2d.text(
        -0.97, 0.95,
        "Arrows = epithelial cell migration",
        fontsize=9,
        color='white',
        bbox=dict(boxstyle="round", facecolor="black", alpha=0.55)
    )

    ax2d.text(
        -0.97, 0.82,
        "Heatmap = proliferation intensity\nP = r_u u(1-u)",
        fontsize=9,
        color='white',
        bbox=dict(boxstyle="round", facecolor="black", alpha=0.55)
    )

    ax2d.text(
        -0.97, 0.66,
        "Black contour = wound boundary",
        fontsize=9,
        color='white',
        bbox=dict(boxstyle="round", facecolor="black", alpha=0.55)
    )

    return []

# ------------------------------------------------------------
# 10. Create animation
# ------------------------------------------------------------
anim = FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=110,
    blit=False
)

plt.close(fig)

# Display animation inside Colab
HTML(anim.to_html5_video())

Simulation finished. Saved 120 frames.
